# Sort with_reprod_annot Files by Verdict

For each of the 20 batches in `sorted_batches_20_reprod_annot/`, this notebook reads
`species_by_verdict.json` and copies every JSON file from `with_reprod_annot/` into one
of two new sub-folders created inside the same batch folder:

| Destination folder | Condition |
|---|---|
| `with_errors/` | species appears in `species_with_errors` |
| `without_errors/` | species appears in `species_without_errors` |

Species identity is derived from the filename: `NNNN_Genus_species_xxx.json` → `Genus_species`.

**Run cells in order: Cell 1 (config) → Cell 2 (main loop) → Cell 3 (summary).**

## Cell 1 — Imports and Configuration

In [1]:
import json
import re
import shutil
from pathlib import Path

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
DATA_ROOT        = Path("../sorted_batches_20_reprod_annot")
N_BATCHES        = 20
VERDICT_FILENAME = "species_by_verdict.json"
WITH_ERRORS_DIR  = "with_errors"
WITHOUT_ERRORS_DIR = "without_errors"

# Filename format: NNNN_Genus_species_xxx.json
# Extract everything between the first underscore group and the last underscore group.
# e.g. "0003_Ablennes_hians_abj.json" -> "Ablennes_hians"
SPECIES_RE = re.compile(r'^\d+_(.+)_[^_]+$')

def species_from_filename(stem: str) -> str | None:
    """Extract 'Genus_species' from a filename stem like '0003_Ablennes_hians_abj'."""
    m = SPECIES_RE.match(stem)
    return m.group(1) if m else None

print(f"Data root : {DATA_ROOT.resolve()}")
print(f"Batches   : {N_BATCHES}")
print(f"Verdict file : {VERDICT_FILENAME}")

# Quick check of the species extraction
for test in ["0002_Abbottina_rivularis_abj", "0003_Ablennes_hians_abj", "0009_Abroscopus_superciliaris_std"]:
    print(f"  {test!r:45s} -> {species_from_filename(test)!r}")

Data root : /Users/somaya/Documents/Phd/github_version/version_2/deb-data-pipeline-github/data_verification/sorted_batches_20_reprod_annot
Batches   : 20
Verdict file : species_by_verdict.json
  '0002_Abbottina_rivularis_abj'                -> 'Abbottina_rivularis'
  '0003_Ablennes_hians_abj'                     -> 'Ablennes_hians'
  '0009_Abroscopus_superciliaris_std'           -> 'Abroscopus_superciliaris'


## Cell 2 — Main Loop

For each batch:
1. Reads `species_by_verdict.json` to build the two species sets.
2. Creates `with_errors/` and `without_errors/` inside the batch folder.
3. Iterates over every `.json` file in `with_reprod_annot/` and copies it to the
   appropriate folder based on the verdict lookup.
4. Files whose species name cannot be matched to either list are flagged as unmatched.

In [2]:
grand_with_errors    = 0
grand_without_errors = 0
grand_unmatched      = 0

batch_stats = []

for batch_num in range(1, N_BATCHES + 1):
    batch_name = f"batch_{batch_num:02d}"
    batch_dir  = DATA_ROOT / batch_name
    src_dir    = batch_dir / "with_reprod_annot"
    verdict_fp = batch_dir / VERDICT_FILENAME

    if not src_dir.exists():
        print(f"[WARN] {src_dir} not found — skipping")
        continue
    if not verdict_fp.exists():
        print(f"[WARN] {verdict_fp} not found — skipping {batch_name}")
        continue

    # Load verdict lists
    verdict = json.loads(verdict_fp.read_text(encoding="utf-8"))
    with_errors_set    = set(verdict.get("species_with_errors", []))
    without_errors_set = set(verdict.get("species_without_errors", []))

    # Create destination folders
    dst_with    = batch_dir / WITH_ERRORS_DIR
    dst_without = batch_dir / WITHOUT_ERRORS_DIR
    dst_with.mkdir(exist_ok=True)
    dst_without.mkdir(exist_ok=True)

    n_with = n_without = n_unmatched = 0
    unmatched_files = []

    for fp in sorted(src_dir.glob("*.json")):
        species = species_from_filename(fp.stem)

        if species in with_errors_set:
            shutil.copy2(fp, dst_with / fp.name)
            n_with += 1
        elif species in without_errors_set:
            shutil.copy2(fp, dst_without / fp.name)
            n_without += 1
        else:
            unmatched_files.append(fp.name)
            n_unmatched += 1

    if unmatched_files:
        print(f"[{batch_name}] {n_unmatched} unmatched file(s):")
        for f in unmatched_files:
            print(f"    {f}")

    grand_with_errors    += n_with
    grand_without_errors += n_without
    grand_unmatched      += n_unmatched

    batch_stats.append({
        "batch":        batch_name,
        "with_errors":  n_with,
        "without_errors": n_without,
        "unmatched":    n_unmatched,
    })

    print(
        f"{batch_name}: {n_with:4d} → {WITH_ERRORS_DIR}/  |  "
        f"{n_without:4d} → {WITHOUT_ERRORS_DIR}/  |  "
        f"{n_unmatched} unmatched"
    )

print(
    f"\nGrand total: {grand_with_errors} with_errors | "
    f"{grand_without_errors} without_errors | "
    f"{grand_unmatched} unmatched"
)

batch_01:  259 → with_errors/  |    59 → without_errors/  |  0 unmatched
batch_02:  246 → with_errors/  |    67 → without_errors/  |  0 unmatched
batch_03:  260 → with_errors/  |    63 → without_errors/  |  0 unmatched
batch_04:  252 → with_errors/  |    48 → without_errors/  |  0 unmatched
batch_05:  267 → with_errors/  |    58 → without_errors/  |  0 unmatched
batch_06:  231 → with_errors/  |    72 → without_errors/  |  0 unmatched
batch_07:  239 → with_errors/  |    75 → without_errors/  |  0 unmatched
batch_08:  266 → with_errors/  |    63 → without_errors/  |  0 unmatched
batch_09:  250 → with_errors/  |    57 → without_errors/  |  0 unmatched
batch_10:  243 → with_errors/  |    63 → without_errors/  |  0 unmatched
batch_11:  248 → with_errors/  |    72 → without_errors/  |  0 unmatched
batch_12:  264 → with_errors/  |    64 → without_errors/  |  0 unmatched
batch_13:  238 → with_errors/  |    58 → without_errors/  |  0 unmatched
batch_14:  255 → with_errors/  |    66 → without_er

## Cell 3 — Summary Table

In [3]:
header = f"{'Batch':<12} {'with_errors':>12} {'without_errors':>15} {'unmatched':>10}"
print(header)
print("-" * len(header))

t_with = t_without = t_unmatched = 0
for s in batch_stats:
    flag = "  <-- CHECK" if s["unmatched"] else ""
    print(
        f"{s['batch']:<12} {s['with_errors']:>12} {s['without_errors']:>15} "
        f"{s['unmatched']:>10}{flag}"
    )
    t_with      += s["with_errors"]
    t_without   += s["without_errors"]
    t_unmatched += s["unmatched"]

print("-" * len(header))
print(f"{'TOTAL':<12} {t_with:>12} {t_without:>15} {t_unmatched:>10}")

if t_unmatched == 0:
    print("\nAll files matched — no unresolved species.")

Batch         with_errors  without_errors  unmatched
----------------------------------------------------
batch_01              259              59          0
batch_02              246              67          0
batch_03              260              63          0
batch_04              252              48          0
batch_05              267              58          0
batch_06              231              72          0
batch_07              239              75          0
batch_08              266              63          0
batch_09              250              57          0
batch_10              243              63          0
batch_11              248              72          0
batch_12              264              64          0
batch_13              238              58          0
batch_14              255              66          0
batch_15              232              70          0
batch_16              268              54          0
batch_17              227              77     